# BS-Detector: birth scars, poles and growth curves from a time-lapse movie

This notebook runs the whole pipeline on one brightfield movie in your Google Drive:

1. **segment** every frame with Cellpose-SAM (the slow step; uses Colab's GPU),
2. **track** cells, assign old and new poles and find each cell's birth scar,
3. **growth**: growth curves for every cell born and dividing in the movie.

**Before you start:** *Runtime > Change runtime type > T4 GPU*. Copy your movie
(`.nd2`, SlideBook `.sldy` + `.dir`, `.tif` or `.h5`) into Google Drive.

**If Colab disconnects,** run all cells again. Masks are saved to Drive as they are made,
so it continues where it stopped. A 9-hour movie of 2304 × 2304 frames takes roughly 1–3
hours to segment on a T4; tracking takes 5–60 minutes.

Leave `MOVIE` empty to try the bundled 5-frame sample. It is too short for complete cell
cycles, so it gives measurements and QC figures but no growth curves.

Guide: [docs/USER_GUIDE.md](https://github.com/widenerm/pombe-bs-detector/blob/main/docs/USER_GUIDE.md)

In [ ]:
#@title 1. Install BS-Detector (about 2 minutes)
!nvidia-smi -L || echo "No GPU: Runtime > Change runtime type > T4 GPU, then run this cell again"
!pip install -q "bs-detector @ git+https://github.com/widenerm/pombe-bs-detector"
!bsd --help | head -3

In [ ]:
#@title 2. Choose the movie
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

MOVIE = ""  #@param {type:"string"}
#@markdown Path in Drive, e.g. `/content/drive/MyDrive/movies/11_19_2025_rgf1DEL_gpd1DEL_YE_.nd2`. Empty = sample movie.
OUT = ""  #@param {type:"string"}
#@markdown Results folder. Empty = `MyDrive/bsd_runs/<movie name>`.
POSITION = ""  #@param {type:"string"}
#@markdown Only for multi-position files; empty = every position, each in its own folder.
HAND = ""  #@param {type:"string"}
#@markdown Optional: hand-tracked growth workbook (.xlsx) to plot next to the pipeline.
HAND_TYPE = ""  #@param {type:"string"}
#@markdown Which cells in the workbook, e.g. `y_wt` (YE wild type) or `y_rgf1delgpd1del`.

if not MOVIE:
    MOVIE = '/content/sample.h5'
    !wget -q -O {MOVIE} https://github.com/widenerm/pombe-bs-detector/raw/main/data/sample/sample.h5
OUT = OUT or f'/content/drive/MyDrive/bsd_runs/{Path(MOVIE).stem}'
pos = f'--position "{POSITION}"' if POSITION else ''
hand = f'--hand "{HAND}"' + (f' --hand-type "{HAND_TYPE}"' if HAND_TYPE else '') if HAND else ''
print('movie:', MOVIE); print('results:', OUT)
!bsd info "{MOVIE}" {pos}

In [ ]:
#@title 3. Segment (GPU; resumable)
!bsd segment "{MOVIE}" --out "{OUT}" {pos}

In [ ]:
#@title 4. Track cells, assign poles, find birth scars
!bsd track "{OUT}"

In [ ]:
#@title 5. Growth curves
!bsd growth "{OUT}" {hand}

In [ ]:
#@title 6. Look at the results
import pandas as pd
from IPython.display import Image, Markdown, display
runs = [Path(OUT)] if (Path(OUT) / 'run.json').exists() else sorted(p.parent for p in Path(OUT).glob('*/run.json'))
for run in runs:
    display(Markdown(f'## {run.name}'))
    m = pd.read_csv(run / 'measurements.csv')
    display(Markdown(f"{m.cell_name.nunique()} cells, {len(m)} cell-frames; "
                     f"{m.scar_needs_review.mean():.0%} of cell-frames flagged for review"))
    display(m.head())
    for png in sorted((run / 'qc').glob('frame_*.png'))[:2]:
        display(Image(str(png), width=600))
    for name in ('growth_since_birth.png', 'growth_since_birth_vs_hand.png'):
        if (run / 'growth' / name).exists():
            display(Image(str(run / 'growth' / name), width=700))
    if (run / 'growth' / 'growth_curves_summary.csv').exists():
        display(pd.read_csv(run / 'growth' / 'growth_curves_summary.csv'))
print('Everything is saved in', OUT)

## What the files are

- `measurements.csv`: one row per cell per frame. Lengths in px and µm; `new_end_length_um`
  and `old_end_length_um` are the two ends; check rows with `scar_needs_review = True`.
- `qc/`: frame overviews and the lineage tree. Look at them before using the numbers.
- `growth/`: figures and tables for every complete cell cycle.
- `run.json`: input, pixel size, versions and settings, for your methods section.

Column-by-column descriptions: [user guide, section 5](https://github.com/widenerm/pombe-bs-detector/blob/main/docs/USER_GUIDE.md#5-what-you-get).